# Retrieval / pencarian dokumen

Jawaban adalah ranking doc_id, bisa banyak dokumen relevan. Group query, cutoff K dan definisi relevance mengikuti soal.

Kode lengkap dari pipeline dan snippet asli. Ikuti urutan tahap dan ubah konfigurasi awal sesuai data. Alternatif tersedia pada cheatsheet utama.

## Setup & konfigurasi

Satu-satunya sel yang wajib diubah. Jalankan paling awal.

In [ ]:
import os
os.environ.update(HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1", TOKENIZERS_PARALLELISM="false")
from pathlib import Path
import importlib.metadata as md

def env_report(models_dir=None):
    import torch
    for p in ("torch", "transformers", "scikit-learn", "pandas", "numpy", "sentence-transformers", "lightgbm"):
        try:
            print(p, md.version(p))
        except md.PackageNotFoundError:
            print(p, "TIDAK ADA")
    gpu = torch.cuda.is_available()
    print("cuda", gpu, torch.cuda.get_device_name(0) if gpu else "")
    if gpu:
        print("VRAM GB", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "| bf16", torch.cuda.is_bf16_supported())
    if models_dir and Path(models_dir).exists():
        print("model lokal:", sorted(p.name for p in Path(models_dir).iterdir() if p.is_dir()))
    return gpu

import os
import random
import numpy as np
import torch

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

# ===== jalankan =====
import re, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
MODELS = Path(os.environ.get("COMP_ROOT", r"C:\gemastik")) / "models"
SEED = 42
seed_everything(SEED)
env_report(MODELS)

DOCS, QUERIES = "docs.csv", "queries.csv"       # kolom: doc_id,text dan query_id,text,relevant_id
K = 10


## Muat data

Baca file, otomatis mencoba encoding, simpan parquet.

In [ ]:
from pathlib import Path
import pandas as pd

def read_any(path, cache=True, **kw):
    """df = read_any('train.csv'). Lain kali otomatis membaca train.parquet."""
    p = Path(path)
    if p.suffix == ".parquet":
        return pd.read_parquet(p, **kw)
    if cache and p.with_suffix(".parquet").exists():
        return pd.read_parquet(p.with_suffix(".parquet"))
    if p.suffix in (".jsonl", ".json"):
        return pd.read_json(p, lines=p.suffix == ".jsonl", **kw)
    if p.suffix in (".xlsx", ".xls"):
        return pd.read_excel(p, **kw)
    sep = kw.pop("sep", "\t" if p.suffix == ".tsv" else ",")
    for enc in ("utf-8", "utf-8-sig", "latin-1"):
        try:
            df = pd.read_csv(p, sep=sep, encoding=enc, on_bad_lines="warn", **kw)
            break
        except UnicodeDecodeError:
            continue
    if cache:
        df.to_parquet(p.with_suffix(".parquet"), index=False)
    return df

# ===== jalankan =====
docs, q = read_any(DOCS), read_any(QUERIES)
dt, qt = docs.text.fillna("").astype(str).tolist(), q.text.fillna("").astype(str).tolist()
doc_ids = docs.doc_id.to_numpy()
print(docs.shape, q.shape)


## Preprocessing

Query dan dokumen harus sama perlakuannya.

In [ ]:
import html
import re
import unicodedata

SLANG = {"yg": "yang", "gk": "tidak", "ga": "tidak", "gak": "tidak", "nggak": "tidak", "tdk": "tidak", "bgt": "banget", "tp": "tapi", "klo": "kalau", "kalo": "kalau",
         "udh": "sudah", "dah": "sudah", "dpt": "dapat", "jg": "juga", "jd": "jadi", "bnyk": "banyak", "karna": "karena", "krn": "karena", "utk": "untuk", "jgn": "jangan",
         "org": "orang", "dr": "dari", "lgi": "lagi", "lg": "lagi", "bnr": "benar", "sdh": "sudah", "dgn": "dengan", "aja": "saja"}
STOPWORDS = set("""ada adalah agar akan aku anda antara apa apakah atau bagi bahwa banyak beberapa begitu bila bisa dalam dan dari dengan di dia dua hal hanya harus ia ini itu
jadi jika juga kami kamu karena ke kepada ketika kita lagi lain lebih maka masih mereka mungkin nya oleh pada para saat saja sama sampai sangat satu saya sebagai sebuah sedang
sehingga sekali selalu semua seperti setelah sudah supaya tapi telah tentang tersebut tetapi untuk yaitu yang""".split())   # sengaja tanpa tidak, bukan, jangan, belum, kurang
NEGATION = "tidak|tak|bukan|jangan|belum|kurang|tanpa"
_MOJIBAKE = re.compile("[\u00c3\u00c2\u00e2\u00f0]")

PRESETS = {
    # keep = biarkan, token = ganti penanda, remove = hapus
    "none":        dict(unicode=False, html=False, url="keep", email="keep", phone="keep", mention="keep", hashtag="keep", emoji="keep", number="keep", lower=False, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
    "tfidf":       dict(unicode=True, html=True, url="token", email="token", phone="token", mention="token", hashtag="keep", emoji="keep", number="keep", lower=True, repeat=True, punct="keep", slang=False, negation=False, stopword=False),
    "social":      dict(unicode=True, html=True, url="token", email="token", phone="token", mention="token", hashtag="word", emoji="text", number="keep", lower=True, repeat=True, punct="keep", slang=True, negation=False, stopword=False),
    "topic":       dict(unicode=True, html=True, url="remove", email="remove", phone="remove", mention="remove", hashtag="word", emoji="remove", number="remove", lower=True, repeat=True, punct="remove", slang=True, negation=False, stopword=True),
    "transformer": dict(unicode=True, html=True, url="keep", email="keep", phone="keep", mention="keep", hashtag="keep", emoji="keep", number="keep", lower=False, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
    "indobertweet": dict(unicode=True, html=True, url="HTTPURL", email="keep", phone="keep", mention="@USER", hashtag="keep", emoji="text", number="keep", lower=True, repeat=False, punct="keep", slang=False, negation=False, stopword=False),
}

def _sub(text, pattern, how, token):
    return text if how == "keep" else re.sub(pattern, " " if how == "remove" else f" {token if how == 'token' else how} ", text)

def preprocess(text, preset="tfidf", slang_dict=SLANG, stopwords=STOPWORDS, **override):
    """preprocess('Halo http://a.id', 'social'). Ubah satu langkah: preprocess(t, 'tfidf', emoji='remove')."""
    c = {**PRESETS[preset], **override}
    t = "" if text is None or text != text else str(text)
    if c["unicode"]:
        if _MOJIBAKE.search(t):                         # teks rusak encoding
            import ftfy
            t = ftfy.fix_text(t)
        t = unicodedata.normalize("NFKC", t).replace("​", "")
    if c["html"]:
        t = re.sub(r"<[^>]+>", " ", html.unescape(t))
    t = _sub(t, r"[\w.+-]+@[\w-]+\.[\w.-]+", c["email"], "EMAIL")          # email sebelum mention
    t = _sub(t, r"https?://\S+|www\.\S+", c["url"], "URL")
    t = _sub(t, r"(?<!\d)(?:\+?62|0)8\d{7,11}(?!\d)", c["phone"], "TELEPON")
    t = _sub(t, r"@\w+", c["mention"], "USER")
    if c["hashtag"] == "word":
        t = re.sub(r"#(\w+)", r" \1 ", t)
    elif c["hashtag"] != "keep":
        t = _sub(t, r"#\w+", c["hashtag"], "HASHTAG")
    if c["emoji"] != "keep":
        import emoji
        if c["emoji"] == "id":                          # emoji jadi kata Indonesia
            from indoNLP.preprocessing import emoji_to_words
            t = emoji_to_words(t, delimiter=(" ", " "))
        else:
            t = emoji.demojize(t, delimiters=(" ", " ")) if c["emoji"] == "text" else emoji.replace_emoji(t, " " if c["emoji"] == "remove" else " EMOJI ")
    t = _sub(t, r"\d+(?:[.,]\d+)*", c["number"], "ANGKA")
    if c["lower"]:
        keep = {m: m for m in ("HTTPURL", "@USER", "URL", "EMAIL", "TELEPON", "USER", "ANGKA", "EMOJI", "HASHTAG") if m in t}
        t = t.lower()
        for m in keep:
            t = re.sub(r"(?<!\w)" + re.escape(m.lower()) + r"(?!\w)", m, t)
    if c["repeat"]:
        t = re.sub(r"(.)\1{2,}", r"\1\1", t)
    if c["punct"] == "remove":
        t = re.sub(r"[^\w\s]", " ", t)
    if c["slang"]:
        t = re.sub(r"\b\w+\b", lambda m: slang_dict.get(m.group(0).lower(), m.group(0)), t)
    if c["negation"]:
        t = re.sub(rf"\b({NEGATION})\s+(\w+)", r"\1_\2", t, flags=re.I)
    if c["stopword"]:
        t = " ".join(w for w in t.split() if w.lower() not in stopwords)
    return re.sub(r"\s+", " ", t).strip()

def preprocess_corpus(texts, preset="tfidf", stem=False, slang_csv=None, **override):
    """Versi cepat: teks identik hanya diproses sekali, stem per kata unik. Kembalikan list."""
    slang = dict(SLANG)
    if slang_csv:
        import pandas as pd
        lex = pd.read_csv(slang_csv)
        slang.update(dict(zip(lex["slang"].astype(str), lex["formal"].astype(str))))
    texts = list(texts)
    cache = {t: preprocess(t, preset, slang_dict=slang, **override) for t in {x if x == x else None for x in texts}}
    out = [cache[t if t == t else None] for t in texts]
    if stem:
        from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
        st = StemmerFactory().create_stemmer()
        m = {w: st.stem(w) for w in {w for d in out for w in d.split()}}
        out = [" ".join(m[w] for w in d.split()) for d in out]
    return out

def text_versions(df, text="text", slang_csv=None):
    """Tiga versi agar efek normalisasi bisa diuji: text_raw, text_clean (makna utuh), text_norm (slang dinormalkan)."""
    d = df.copy()
    d["text_raw"] = d[text]
    d["text_clean"] = preprocess_corpus(d[text], "tfidf")
    d["text_norm"] = preprocess_corpus(d[text], "social", slang_csv=slang_csv)
    return d

# ===== jalankan =====
dt, qt = preprocess_corpus(dt, "tfidf"), preprocess_corpus(qt, "tfidf")


## Retrieval

BM25, TF-IDF, lalu digabung RRF. E5 bisa ditambahkan sebagai ranking ketiga.

In [ ]:
import numpy as np
from scipy import sparse
from sklearn.feature_extraction.text import CountVectorizer

class BM25:
    def __init__(self, k1=1.5, b=.75, ngram=(1, 1)):
        self.k1, self.b, self.v = k1, b, CountVectorizer(ngram_range=ngram, token_pattern=r"(?u)\b\w+\b")
    def fit(self, docs):
        X = self.v.fit_transform(docs).astype("float32").tocsr()
        n, dl = X.shape[0], np.asarray(X.sum(1)).ravel()
        df_ = np.asarray((X > 0).sum(0)).ravel()
        self.idf_ = np.log(1 + (n - df_ + .5) / (df_ + .5))
        rows = np.repeat(np.arange(n), np.diff(X.indptr))
        d = X.data * (self.k1 + 1) / (X.data + self.k1 * (1 - self.b + self.b * dl[rows] / dl.mean()))
        self.M_ = sparse.csr_matrix((d * self.idf_[X.indices], X.indices, X.indptr), shape=X.shape)
        return self
    def scores(self, queries):
        return (self.v.transform(queries).astype("float32") > 0).astype("float32") @ self.M_.T
    def search(self, queries, k=10):
        """Kembalikan (idx, skor) top-k per query. Batch per 1.000 query pada korpus besar."""
        S = self.scores(queries).toarray()
        idx = np.argsort(-S, 1)[:, :k]
        return idx, np.take_along_axis(S, idx, 1)

import numpy as np
from sklearn.metrics.pairwise import linear_kernel

def topk_dense(Q, D, k=10, batch=512):
    """Q dan D sudah L2-normalized. Kembalikan (idx, skor)."""
    D = np.asarray(D); idx, sc = [], []
    for i in range(0, len(Q), batch):
        S = np.asarray(Q[i:i + batch]) @ D.T
        p = np.argpartition(-S, min(k, D.shape[0] - 1), 1)[:, :k]
        s = np.take_along_axis(S, p, 1)
        o = np.argsort(-s, 1)
        idx.append(np.take_along_axis(p, o, 1)); sc.append(np.take_along_axis(s, o, 1))
    return np.vstack(idx), np.vstack(sc)

def topk_sparse(Q, D, k=10, batch=512):
    """TF-IDF kosinus top-k per batch."""
    idx, sc = [], []
    for i in range(0, Q.shape[0], batch):
        S = linear_kernel(Q[i:i + batch], D)
        p = np.argsort(-S, 1)[:, :k]
        idx.append(p); sc.append(np.take_along_axis(S, p, 1))
    return np.vstack(idx), np.vstack(sc)

def rrf_merge(rankings, k=60, top=10):
    """rankings: list matriks idx (n_query, n_kandidat). Reciprocal Rank Fusion."""
    out = []
    for q in range(rankings[0].shape[0]):
        s = {}
        for r in rankings:
            for pos, d in enumerate(r[q]):
                s[int(d)] = s.get(int(d), 0) + 1 / (k + pos + 1)
        out.append(sorted(s, key=s.get, reverse=True)[:top])
    return np.array(out)

import os
import re
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer, TfidfVectorizer
from sklearn.pipeline import make_pipeline, make_union
from sklearn.preprocessing import FunctionTransformer, Normalizer

def _tokens(t):
    return re.findall(r"\w+", str(t).lower())

def _meta(texts):
    s = pd.Series(list(texts)).fillna("").astype(str)
    n = s.str.len().clip(lower=1)
    return sparse.csr_matrix(pd.DataFrame({"len": np.log1p(n), "words": np.log1p(s.str.split().str.len()), "upper": s.str.count(r"[A-Z]") / n, "digit": s.str.count(r"\d") / n,
                                           "punct": s.str.count(r"[^\w\s]") / n, "excl": s.str.count("!").clip(upper=5), "url": s.str.count(r"https?://|URL"), "mention": s.str.count(r"@\w+|USER")}).to_numpy("float32"))

class RawText(BaseEstimator, TransformerMixin):
    """Teks mentah untuk model neural."""
    def fit(self, X, y=None): return self
    def transform(self, X): return list(X)

class GensimVec(BaseEstimator, TransformerMixin):
    """kind: w2v, fasttext, doc2vec. idf=True: rata-rata berbobot."""
    def __init__(self, kind="w2v", dim=200, window=5, min_count=2, epochs=5, idf=False, workers=4, seed=42):
        self.kind, self.dim, self.window, self.min_count, self.epochs, self.idf, self.workers, self.seed = kind, dim, window, min_count, epochs, idf, workers, seed
    def fit(self, X, y=None):
        from gensim.models import FastText, Word2Vec
        from gensim.models.doc2vec import Doc2Vec, TaggedDocument
        toks = [_tokens(t) for t in X]
        kw = dict(vector_size=self.dim, min_count=self.min_count, epochs=self.epochs, workers=self.workers, seed=self.seed)
        if self.kind == "doc2vec":
            self.m_ = Doc2Vec([TaggedDocument(t, [i]) for i, t in enumerate(toks)], dm=0, **kw)
        else:
            self.m_ = (FastText if self.kind == "fasttext" else Word2Vec)(toks, window=self.window, **kw).wv
        self.w_ = None
        if self.idf:
            t = TfidfVectorizer(token_pattern=r"\w+").fit(X)
            self.w_ = dict(zip(t.get_feature_names_out(), t.idf_))
        return self
    def transform(self, X):
        out = np.zeros((len(X), self.dim), dtype="float32")
        for i, t in enumerate(X):
            w = _tokens(t)
            if self.kind == "doc2vec":
                out[i] = self.m_.infer_vector(w)
                continue
            w = w if self.kind == "fasttext" else [x for x in w if x in self.m_]
            if w:
                a = np.array([self.w_.get(x, 1.0) for x in w])[:, None] if self.w_ else np.ones((len(w), 1))
                out[i] = (self.m_[w] * a).sum(0) / a.sum()
        return out

class EmbedVec(BaseEstimator, TransformerMixin):
    """Embedding SentenceTransformer dengan cache per teks agar tiap fold tidak encode ulang."""
    _cache = {}
    def __init__(self, model_path=None, prefix="", batch=128):
        self.model_path, self.prefix, self.batch = model_path, prefix, batch
    def fit(self, X, y=None): return self
    def transform(self, X):
        c = EmbedVec._cache.setdefault(str(self.model_path), {})
        miss = [t for t in dict.fromkeys(map(str, X)) if t not in c]
        if miss:
            from sentence_transformers import SentenceTransformer
            m = SentenceTransformer(str(self.model_path), local_files_only=True)
            for t, v in zip(miss, m.encode([self.prefix + t for t in miss], batch_size=self.batch, normalize_embeddings=True, show_progress_bar=False)):
                c[t] = v.astype("float32")
        return np.vstack([c[str(t)] for t in X])

def make_vectorizer(kind="tfidf_union", **kw):
    """Semua hasilnya punya fit_transform dan transform."""
    word = dict(ngram_range=(1, 2), min_df=2, max_df=.95, sublinear_tf=True, dtype="float32", token_pattern=r"(?u)\b\w+\b")
    char = dict(analyzer="char_wb", ngram_range=(2, 5), min_df=3, max_features=500_000, sublinear_tf=True, dtype="float32")
    if kind == "tfidf_word": return TfidfVectorizer(**{**word, **kw})
    if kind == "tfidf_char": return TfidfVectorizer(**{**char, **kw})
    if kind == "tfidf_union": return make_union(TfidfVectorizer(**word), TfidfVectorizer(**char))
    if kind == "count": return CountVectorizer(**{**dict(ngram_range=(1, 1), min_df=2, max_df=.95), **kw})
    if kind == "binary": return CountVectorizer(**{**dict(ngram_range=(1, 2), min_df=2, binary=True), **kw})
    if kind == "hashing": return HashingVectorizer(**{**dict(n_features=2 ** 20, alternate_sign=False, ngram_range=(1, 2), dtype="float32"), **kw})
    if kind == "lsa": return make_pipeline(TfidfVectorizer(**word), TruncatedSVD(kw.pop("k", 256), random_state=42), Normalizer(copy=False))
    if kind == "meta": return FunctionTransformer(_meta, validate=False)
    if kind == "tfidf_meta": return make_union(TfidfVectorizer(**word), FunctionTransformer(_meta, validate=False))
    if kind in ("w2v", "fasttext", "doc2vec"): return GensimVec(kind, **kw)
    if kind == "w2v_idf": return GensimVec("w2v", idf=True, **kw)
    if kind == "embed": return EmbedVec(**kw)
    if kind == "raw": return RawText()
    raise ValueError(f"vectorizer tidak dikenal: {kind}")

# ===== jalankan =====
bm, _ = BM25().fit(dt).search(qt, K * 5)
vec = make_vectorizer("tfidf_word", min_df=1)
Dm = vec.fit_transform(dt)
tf, _ = topk_sparse(vec.transform(qt), Dm, K * 5)
fused = rrf_merge([bm, tf], top=K * 5)


## Evaluasi

Bila ada relevant_id.

In [ ]:
import numpy as np

def retrieval_report(ranked, relevant, k=10):
    """ranked: list daftar id terurut per query. relevant: list set id (atau dict id->gain untuk NDCG)."""
    P, R, AP, RR, N = [], [], [], [], []
    for r, rel in zip(ranked, relevant):
        gain = rel if isinstance(rel, dict) else {x: 1 for x in rel}
        rs = [x for x in gain if gain[x] > 0]
        hit = [x in rs for x in r]
        P.append(sum(hit[:k]) / k)
        R.append(sum(hit[:k]) / max(len(rs), 1))
        c = np.cumsum(hit)
        AP.append(float(np.sum([c[i] / (i + 1) for i, h in enumerate(hit) if h])) / max(len(rs), 1))
        RR.append(next((1 / (i + 1) for i, h in enumerate(hit) if h), 0))
        dcg = sum((2 ** gain.get(x, 0) - 1) / np.log2(i + 2) for i, x in enumerate(r[:k]))
        idcg = sum((2 ** g - 1) / np.log2(i + 2) for i, g in enumerate(sorted(gain.values(), reverse=True)[:k]))
        N.append(dcg / idcg if idcg else 0)
    return {f"p@{k}": round(float(np.mean(P)), 4), f"r@{k}": round(float(np.mean(R)), 4), "map": round(float(np.mean(AP)), 4), "mrr": round(float(np.mean(RR)), 4), f"ndcg@{k}": round(float(np.mean(N)), 4)}

def keyword_f1_at_k(pred, gold, k=5):
    """pred: list daftar kata kunci. gold: list daftar emas. Pencocokan ternormalisasi."""
    tp = np.array([len({x.lower() for x in p[:k]} & {x.lower() for x in g}) for p, g in zip(pred, gold)])
    prec = tp.sum() / max(sum(min(k, len(p)) for p in pred), 1)
    rec = tp.sum() / max(sum(len(g) for g in gold), 1)
    return round(2 * prec * rec / max(prec + rec, 1e-9), 4)

# ===== jalankan =====
gold = [{x} for x in q.relevant_id] if "relevant_id" in q else None
if gold:
    for n, r in (("bm25", bm), ("tfidf", tf), ("rrf", fused)):
        print(n, retrieval_report([doc_ids[x].tolist() for x in r], gold, K))


## Re-ranking

LGBMRanker dengan fitur peringkat, bila ada relevansi.

In [ ]:
import lightgbm as lgb
import numpy as np

def fit_ranker(X, rel, qid, n=300, lr=.05):
    qid = np.asarray(qid)
    order = np.argsort(qid, kind="stable")
    _, sizes = np.unique(qid[order], return_counts=True)
    m = lgb.LGBMRanker(objective="lambdarank", n_estimators=n, learning_rate=lr, verbose=-1, min_child_samples=1)
    m.fit(np.asarray(X)[order], np.asarray(rel)[order], group=sizes)
    return m

# ===== jalankan =====
final = fused[:, :K]
if gold:
    rows = []
    for i, c in enumerate(fused):
        for rk, d in enumerate(c):
            rows.append((i, int(d), rk, int(np.where(bm[i] == d)[0][0]) if d in bm[i] else 99, int(np.where(tf[i] == d)[0][0]) if d in tf[i] else 99, int(doc_ids[d] in gold[i])))
    r = pd.DataFrame(rows, columns=["q", "d", "rrf", "bm", "tf", "rel"])
    fit = r.q % 2 == 0                       # latih pada query genap, nilai pada ganjil
    rk = fit_ranker(r.loc[fit, ["rrf", "bm", "tf"]], r.rel[fit], r.q[fit], 100)
    r["s"] = rk.predict(r[["rrf", "bm", "tf"]])
    rr = r.sort_values(["q", "s"], ascending=[True, False]).groupby("q").d.apply(list)
    odd = [i for i in range(len(q)) if i % 2]
    print("ranker, query ganjil", retrieval_report([doc_ids[rr[i]].tolist() for i in odd], [gold[i] for i in odd], K))


## Output

Satu baris per query.

In [ ]:
q["pred"] = [" ".join(map(str, doc_ids[x])) for x in final]
q[["query_id", "pred"]].to_csv("submission.csv", index=False)
print(q[["query_id", "pred"]].head(3).to_string())
